In [7]:
from pathlib import Path
import json
import math
import warnings
from collections import Counter, defaultdict
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, MultiLabelBinarizer
from sklearn.metrics import (
    accuracy_score, f1_score, mean_absolute_error, mean_squared_error, r2_score,
    silhouette_score
)
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity, pairwise_distances

from IPython.display import display

warnings.filterwarnings("ignore")

In [2]:
# ============================================================
# Project / run configuration
# ============================================================
DEFAULT_PROJECT_ROOT = Path("/home/tungichen_umass_edu/ekb-claude-pilot")
PROJECT_ROOT = DEFAULT_PROJECT_ROOT if DEFAULT_PROJECT_ROOT.exists() else Path.cwd().resolve()

RESULTS_ROOT = PROJECT_ROOT / "results" / "claude_native"

# Change this to the run folder you want to analyze
RUN_NAME = "gaia_level1_expanded_claude_native_maxturn12"

RUN_ROOT = RESULTS_ROOT / RUN_NAME
OUTPUT_DIR = PROJECT_ROOT / "results" / "analysis" / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RUN_ROOT:", RUN_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def load_json(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

def iter_normalized_trace_paths(run_root: Path):
    """
    New logic:
    results/claude_native/<RUN_NAME>/<task_id>/normalized_trace.json

    where <task_id> is an arbitrary folder name.
    """
    if not run_root.exists():
        raise FileNotFoundError(f"Trace root does not exist: {run_root}")

    # Only look one directory below RUN_ROOT
    for path in sorted(run_root.glob("*/normalized_trace.json")):
        yield path

def canonical_tool_name(tool):
    if tool is None:
        return "unknown"
    tool = str(tool).strip()
    mapping = {
        "WebSearch": "WebSearch",
        "WebFetch": "WebFetch",
        "Bash": "Bash",
        "Read": "Read",
        "Write": "Write",
        "Edit": "Edit",
        "Grep": "Grep",
        "Glob": "Glob",
        "ToolSearch": "ToolSearch",
        "StructuredOutput": "StructuredOutput",
    }
    return mapping.get(tool, tool)

def extract_steps(trace: dict):
    steps = trace.get("steps", [])
    if not isinstance(steps, list):
        return []

    out = []
    for s in steps:
        if not isinstance(s, dict):
            continue
        out.append({
            "step": s.get("step"),
            "tool": canonical_tool_name(s.get("tool")),
            "status": s.get("status"),
            "latency_ms": s.get("latency_ms"),
        })
    return out

def safe_num(x, default=np.nan):
    try:
        if x is None:
            return default
        return float(x)
    except Exception:
        return default

def build_trace_record(path: Path):
    trace = load_json(path)
    steps = extract_steps(trace)

    tools_all = [s["tool"] for s in steps]
    tool_counts = Counter(tools_all)

    record = {
        "query_id": trace.get("query_id"),
        "query_text": trace.get("query_text", ""),
        "benchmark": trace.get("benchmark"),
        "split": trace.get("split"),
        "level": trace.get("level"),
        "agent": trace.get("agent"),
        "model_requested": trace.get("model_requested"),
        "effort": trace.get("effort"),

        # scalar execution attributes
        "total_steps": int(trace.get("total_steps", len(steps) or 0)),
        "total_tool_calls": int(trace.get("total_tool_calls", len(steps) or 0)),
        "total_latency_ms": safe_num(trace.get("total_latency_ms")),
        "total_tokens": safe_num(trace.get("total_tokens")),

        # sequence / set views
        "tool_sequence_all": tools_all,
        "tools_used": sorted(set(tools_all)),

        # answer/eval fields (useful later)
        "success": trace.get("success"),
        "final_answer_pred": trace.get("final_answer_pred"),
        "ground_truth_answer": trace.get("ground_truth_answer"),
        "exact_match": trace.get("exact_match"),
        "result_text": trace.get("result_text"),

        # bookkeeping
        "normalized_trace_path": str(path),
        "task_dir": str(path.parent),
        "run_name": path.parent.parent.name,
        "task_folder_name": path.parent.name,

        # keep raw tool counts temporarily for dense expansion later
        "_tool_counts": dict(tool_counts),
    }
    return record

# ============================================================
# Load traces
# ============================================================
trace_paths = list(iter_normalized_trace_paths(RUN_ROOT))
print(f"Found {len(trace_paths)} normalized_trace.json files under {RUN_ROOT}")

trace_records = [build_trace_record(p) for p in trace_paths]
df = pd.DataFrame(trace_records)

print(f"Loaded {len(df)} normalized traces.")

# ------------------------------------------------------------
# Expand dense tool_count__* columns with zeros instead of NaN
# ------------------------------------------------------------
all_tools = sorted({
    tool
    for record in trace_records
    for tool in record["_tool_counts"].keys()
})

for tool in all_tools:
    col = f"tool_count__{tool}"
    df[col] = df["_tool_counts"].apply(lambda d: int(d.get(tool, 0)))

df = df.drop(columns=["_tool_counts"])

# ------------------------------------------------------------
# Ensure scalar columns are numeric
# ------------------------------------------------------------
for c in ["total_steps", "total_tool_calls", "total_latency_ms", "total_tokens"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Optional sanity fill
df["total_steps"] = df["total_steps"].fillna(0).astype(int)
df["total_tool_calls"] = df["total_tool_calls"].fillna(0).astype(int)
df["total_latency_ms"] = df["total_latency_ms"].fillna(0.0)
df["total_tokens"] = df["total_tokens"].fillna(0.0)

# ------------------------------------------------------------
# Show only the columns that matter for later analysis
# ------------------------------------------------------------
tool_count_cols = sorted([c for c in df.columns if c.startswith("tool_count__")])

display_cols = [
    "query_id",
    "query_text",
    "total_steps",
    "total_tool_calls",
    "total_latency_ms",
    "total_tokens",
    "tool_sequence_all",
    "tools_used",
    "success",
    "exact_match",
] + tool_count_cols

display(df[display_cols].head(3))

print("Observed tools:", all_tools)
print("Tool-count columns:", tool_count_cols)

PROJECT_ROOT: /home/tungichen_umass_edu/ekb-claude-pilot
RUN_ROOT: /home/tungichen_umass_edu/ekb-claude-pilot/results/claude_native/gaia_level1_expanded_claude_native_maxturn12
OUTPUT_DIR: /home/tungichen_umass_edu/ekb-claude-pilot/results/analysis/gaia_level1_expanded_claude_native_maxturn12
Found 194 normalized_trace.json files under /home/tungichen_umass_edu/ekb-claude-pilot/results/claude_native/gaia_level1_expanded_claude_native_maxturn12
Loaded 194 normalized traces.


,query_id,query_text,total_steps,total_tool_calls,total_latency_ms,total_tokens,tool_sequence_all,tools_used,success,exact_match,tool_count__Bash,tool_count__Glob,tool_count__Read,tool_count__StructuredOutput,tool_count__ToolSearch,tool_count__WebFetch,tool_count__WebSearch
0,0383a3ee-47a7-41a4-b493-519bdefe0488,On the BBC Earth YouTube video of the Top 5 Si...,9,9,83683.0,204013.0,"[ToolSearch, WebSearch, ToolSearch, WebFetch, ...","[StructuredOutput, ToolSearch, WebFetch, WebSe...",True,False,0,0,0,1,2,4,2
1,0383a3ee-47a7-41a4-b493-519bdefe0488__para1,In the BBC Earth YouTube video titled Top 5 Si...,8,8,49461.0,179148.0,"[ToolSearch, WebSearch, ToolSearch, WebFetch, ...","[StructuredOutput, ToolSearch, WebFetch, WebSe...",True,True,0,0,0,1,2,3,2
2,0383a3ee-47a7-41a4-b493-519bdefe0488__para2,What bird species is shown in the BBC Earth Yo...,11,11,144579.0,249898.0,"[ToolSearch, WebSearch, ToolSearch, WebFetch, ...","[StructuredOutput, ToolSearch, WebFetch, WebSe...",True,True,0,0,0,2,2,4,3


Observed tools: ['Bash', 'Glob', 'Read', 'StructuredOutput', 'ToolSearch', 'WebFetch', 'WebSearch']
Tool-count columns: ['tool_count__Bash', 'tool_count__Glob', 'tool_count__Read', 'tool_count__StructuredOutput', 'tool_count__ToolSearch', 'tool_count__WebFetch', 'tool_count__WebSearch']


In [6]:
# ============================================================
# Compute final-answer accuracy from normalized_trace.json files
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def load_json(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

def iter_normalized_trace_paths(results_root: Path):
    if not results_root.exists():
        raise FileNotFoundError(f"Trace root does not exist: {results_root}")
    for path in sorted(results_root.glob("*/normalized_trace.json")):
        yield path

def normalize_answer(x):
    if x is None:
        return None
    x = str(x).strip()
    if x == "":
        return ""
    return x.casefold()

# ------------------------------------------------------------
# Load answer-level evaluation fields
# ------------------------------------------------------------
answer_records = []

for path in iter_normalized_trace_paths(RUN_ROOT):
    trace = load_json(path)

    pred = trace.get("final_answer_pred")
    gt = trace.get("ground_truth_answer")
    exact = trace.get("exact_match")

    # Fallback if exact_match is missing
    if exact is None:
        pred_norm = normalize_answer(pred)
        gt_norm = normalize_answer(gt)
        exact = (pred_norm == gt_norm) if (pred_norm is not None and gt_norm is not None) else False

    answer_records.append({
        "query_id": trace.get("query_id"),
        "query_text": trace.get("query_text", ""),
        "shard_name": path.parent.parent.name,
        "final_answer_pred": pred,
        "ground_truth_answer": gt,
        "exact_match": bool(exact),
        "success": trace.get("success"),
        "normalized_trace_path": str(path),
    })

df_acc = pd.DataFrame(answer_records)

if len(df_acc) == 0:
    raise ValueError("No normalized_trace.json files were found.")

# ------------------------------------------------------------
# Overall accuracy
# ------------------------------------------------------------
n_total = len(df_acc)
n_correct = int(df_acc["exact_match"].sum())
acc = n_correct / n_total

print(f"Loaded {n_total} traces")
print(f"Correct: {n_correct}")
print(f"Accuracy: {acc:.4f} ({acc*100:.2f}%)")

Loaded 194 traces
Correct: 141
Accuracy: 0.7268 (72.68%)


In [8]:
# ------------------------------------------------------------
# Basic normalization helpers
# ------------------------------------------------------------
def ensure_list(x):
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    return [x]

def normalize_tool_token(t):
    if t is None:
        return "UNKNOWN"
    return str(t).strip()

# ------------------------------------------------------------
# Normalize sequence / set columns
# ------------------------------------------------------------
df["tool_sequence_all"] = df["tool_sequence_all"].apply(
    lambda seq: [normalize_tool_token(t) for t in ensure_list(seq)]
)

# If tools_used_core exists, use it; otherwise fall back to tools_used
if "tools_used_core" in df.columns:
    df["tools_used"] = df["tools_used_core"].apply(
        lambda xs: [normalize_tool_token(x) for x in ensure_list(xs)]
    )
else:
    df["tools_used"] = df["tools_used"].apply(
        lambda xs: [normalize_tool_token(x) for x in ensure_list(xs)]
    )

# ------------------------------------------------------------
# 1) Scalar execution attributes
# ------------------------------------------------------------
scalar_cols = [
    "total_tool_calls",
    "total_latency_ms",
    "total_tokens",
]

scalar_df = df[scalar_cols].copy()
for c in scalar_cols:
    scalar_df[c] = pd.to_numeric(scalar_df[c], errors="coerce")

scalar_df = scalar_df.fillna(0.0)

print("Scalar execution attributes")
display(scalar_df.head(10))

# ------------------------------------------------------------
# 2) Tool-count vector
# ------------------------------------------------------------
tool_count_cols = sorted([c for c in df.columns if c.startswith("tool_count__")])

tool_count_df = df[tool_count_cols].copy()
for c in tool_count_cols:
    tool_count_df[c] = pd.to_numeric(tool_count_df[c], errors="coerce")

tool_count_df = tool_count_df.fillna(0.0)

# ------------------------------------------------------------
# 3) Tool-sequence structure
# ------------------------------------------------------------
seq_obs_df = df[["query_text", "tool_sequence_all"]].copy()
seq_obs_df["tool_sequence_str"] = seq_obs_df["tool_sequence_all"].apply(lambda seq: " -> ".join(seq))

# Also inspect common exact sequences
seq_freq_df = (
    seq_obs_df["tool_sequence_str"]
    .value_counts()
    .rename_axis("tool_sequence_str")
    .reset_index(name="count")
)

print("Most common tool sequences")
display(seq_freq_df.head(10))

print("Toal number of different tool sequences")
print(len(seq_freq_df))

# Tool-set membership as a separate diagnostic attribute
tool_set_obs_df = df[["query_text", "tools_used"]].copy()
tool_set_obs_df["tool_set_str"] = tool_set_obs_df["tools_used"].apply(lambda xs: ", ".join(sorted(set(xs))))

Scalar execution attributes


,total_tool_calls,total_latency_ms,total_tokens
0,9,83683.0,204013.0
1,8,49461.0,179148.0
2,11,144579.0,249898.0
3,10,71740.0,225716.0
4,1,9087.0,35542.0
5,1,5800.0,35540.0
6,1,7309.0,35559.0
7,1,8598.0,35653.0
8,8,32144.0,112226.0
9,24,67092.0,235351.0


Most common tool sequences


,tool_sequence_str,count
0,StructuredOutput,38
1,ToolSearch -> WebSearch -> ToolSearch -> WebFe...,13
2,ToolSearch -> WebSearch -> StructuredOutput,11
3,ToolSearch -> WebSearch -> ToolSearch -> WebFe...,8
4,StructuredOutput -> StructuredOutput,5
5,ToolSearch -> WebSearch -> WebSearch -> Struct...,4
6,Bash -> Bash -> Bash -> Bash -> Bash -> Bash -...,4
7,Bash -> Bash -> Bash -> Bash -> Bash -> Bash -...,4
8,Bash -> StructuredOutput,3
9,ToolSearch -> WebFetch -> WebSearch -> Bash ->...,3


Toal number of different tool sequences
105


In [9]:
# Build embedding matrix for prompt semantics
EMBEDDING_BACKEND = None

def make_embeddings(texts):
    global EMBEDDING_BACKEND
    texts = [str(t) for t in texts]

    try:
        from sentence_transformers import SentenceTransformer
        model_name = "sentence-transformers/all-MiniLM-L6-v2"
        model = SentenceTransformer(model_name)
        X = model.encode(texts, show_progress_bar=True, normalize_embeddings=True)
        EMBEDDING_BACKEND = model_name
        return np.asarray(X, dtype=np.float32)
    except Exception as e:
        print("SentenceTransformer unavailable; falling back to TF-IDF + SVD")
        print("Reason:", repr(e))
        vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=1)
        X_tfidf = vectorizer.fit_transform(texts)
        n_components = min(128, max(2, X_tfidf.shape[1] - 1))
        svd = TruncatedSVD(n_components=n_components, random_state=42)
        X = svd.fit_transform(X_tfidf)
        # normalize rows so cosine similarity behaves sensibly
        norms = np.linalg.norm(X, axis=1, keepdims=True)
        norms[norms == 0] = 1.0
        X = X / norms
        EMBEDDING_BACKEND = "tfidf+svd"
        return np.asarray(X, dtype=np.float32)

X_embed = make_embeddings(df["query_text"].tolist())
print("Embedding backend:", EMBEDDING_BACKEND)
print("Embedding shape:", X_embed.shape)


Batches: 100%|██████████| 7/7 [00:00<00:00,  7.04it/s]

Embedding backend: sentence-transformers/all-MiniLM-L6-v2
Embedding shape: (194, 384)
